In [ ]:
import os
os.chdir(r"C:\Users\Admin\Desktop\hospoda.CZECH-PUB\code\czech-pub-copy-to-load\czech-pub")

In [1]:
from __future__ import annotations

from utils.helpers import load_dataset, normalize

import json
from pathlib import Path
from collections import Counter, defaultdict

In [8]:
# data and file paths
DATA_PATH = Path.cwd() / "data/information-structure/generated"

FILES = [
    "inf-structure-baseline-80.json",
    "inf-structure-explicit-question-80.json",
    "inf-structure-correction-80.json",
]  


### remaking the correction set

In [ ]:
cor_limited = [{"id": item["scenario_id"], "context": item["seed"]["contexts"]["focus_object"], "utterance": item["seed"]["utterance"]} for item in cor]
with open(r"data\information-structure\generated\upd_before_correction.json", "w", encoding="utf-8") as f:
    json.dump(cor_limited, f, ensure_ascii=False, indent=2)


#### generated updated items

In [ ]:
cor_upd = load_dataset(r"data\information-structure\generated\upd_correction.json")
len(cor_upd)

for i in range(80):
    if cor_limited[i]['context'] != cor_upd[i]['context']:
        print()
        print("shit:", cor_limited[i]['id'])

        print(cor_limited[i]['context'])
        print("---")
        print(cor_upd[i]['context'])

In [ ]:
# change the utterances in inf_structure 
for i in range(len(cor_upd)):
    for j in range(len(cor_upd)):
        if cor_upd[i]['id'] == cor[j]['scenario_id']:
            #print(f"new: {cor_upd[j]['utterance']}\nold:{cor[j]['utterance']}")
            cor[j]['seed']['utterance'] = cor_upd[j]['utterance']
cor[0]
    

In [ ]:
cor_path = DATA_PATH / "inf-structure-correction-80-raw.json"
with open(cor_path, "w", encoding="utf-8") as f:
    json.dump(cor, f, ensure_ascii=False, indent=2)

    # in the scenario 153 the triple is: "subject": "Tomáš", "verb": "opravil", "object": "kliku"
    # we want to avoid repetition of "opravit" -> change manually to "upresnit"

## data analysis & check for missing values

In [9]:
all_data = []

for file in FILES:
    p = DATA_PATH / file
    data = load_dataset(p)

    print(f"loaded {file}: {len(data)} scenarios")
    all_data.extend(data)

print("\ntotal scenarios:", len(all_data))

loaded inf-structure-baseline-80.json: 80 scenarios
loaded inf-structure-explicit-question-80.json: 80 scenarios
loaded inf-structure-correction-80.json: 80 scenarios

total scenarios: 240


In [10]:
# cue distribution
cue_counts = Counter(row.get("cue") for row in all_data)
print("Cue counts:", dict(cue_counts))

# do those fields exist: just to check
required_top = {"scenario_id", "cue", "seed"}
missing = []
for i, row in enumerate(all_data, start=1):
    missing_fields = required_top - set(row.keys())
    if missing_fields:
        missing.append((i, row.get("scenario_id"), sorted(missing_fields)))

print("Missing required top-level fields:", missing if missing else "NONE")

Cue counts: {'baseline': 80, 'explicit_question': 80, 'correction': 80}
Missing required top-level fields: NONE


In [11]:
# detect repeating svo triples in seed.base_proposition 

def find_repeated_triples(rows):

    triple_to_items = {}

    for idx, row in enumerate(rows, start=1):
        base = row.get("seed", {}).get("base_proposition", {})
        s, v, o = base.get("subject"), base.get("verb"), base.get("object")
        if s is None or v is None or o is None:
            raise ValueError(f"none in the triples.")

        triple = (normalize(s), normalize(v), normalize(o))
        key = row.get("scenario_id")
        
        if triple not in triple_to_items:
            triple_to_items[triple] = []
        triple_to_items[triple].append(key)

    repeats = {t: ids for t, ids in triple_to_items.items() if len(ids) > 1}
    return repeats, triple_to_items

In [12]:
repeats, all_triples = find_repeated_triples(all_data)
print("unique:", len(all_triples), "repeated:", len(repeats))
for triple, ids in repeats.items():
    print(triple, "->", ids)

unique: 230 repeated: 10
('grafik', 'upravil', 'logo') -> [30, 35]
('skladník', 'vybalil', 'zásilku') -> [31, 2]
('barman', 'namíchal', 'drink') -> [32, 65]
('zahradník', 'ostříhal', 'keře') -> [38, 17]
('údržbář', 'namazal', 'zámek') -> [46, 6]
('automechanik', 'vypustil', 'nádrž') -> [48, 15]
('instalatér', 'utáhl', 'kohoutek') -> [50, 18]
('klára', 'ztratila', 'prsten') -> [62, 111]
('technik', 'restartoval', 'server') -> [64, 36]
('sekretářka', 'naskenovala', 'smlouvu') -> [12, 102]


## explode scenarios to items

In [13]:
def capitalize_first(text: str):
    text = " ".join(str(text).split())
    if not text:
        return text
    return text[0].upper() + text[1:]


def make_sentence(parts, ending="."):
    sentence = " ".join(str(p).strip() for p in parts if p is not None)
    sentence = capitalize_first(sentence)
    if sentence and sentence[-1] not in ".!?":
        sentence += ending
    return sentence


def unpack_basic_proposition(base: dict):
    s = base["subject"]
    v = base["verb"]
    o = base["object"]

    return {
        "SVO": make_sentence([s, v, o]),
        "VSO": make_sentence([v, s, o]),
        "OVS": make_sentence([o, v, s]),
        "SOV": make_sentence([s, o, v]),
    }


def object_last_structure_for_case(case: str):
    """
    object_last option:
    - focus_object -> SVO
    - focus_subject / focus_verb -> VSO
    """
    if case == "focus_object":
        return "SVO"
    if case in {"focus_subject", "focus_verb"}:
        return "VSO"
    raise ValueError(f"Unknown case: {case}")


def right_answer_for_case(case: str) -> str:
    """ map each case to the corresponding right answer """
    mapping = {
        "focus_object": "object_last",
        "focus_subject": "subject_last",
        "focus_verb": "verb_last",
    }
    return mapping[case]


def build_options(seed: dict, case: str) -> dict:
    """ builds answer options for the given case, extracting data from the scenairo's seed """
    variants = unpack_basic_proposition(seed["base_proposition"])
    object_last_structure = object_last_structure_for_case(case)

    return {
        "object_last": {
            "structure": object_last_structure,
            "sentence": variants[object_last_structure],
        },
        "subject_last": {
            "structure": "OVS",
            "sentence": variants["OVS"],
        },
        "verb_last": {
            "structure": "SOV",
            "sentence": variants["SOV"],
        },
        "distractor": {
            "sentence": seed["distractor_bank"][case],
        },
    }


def build_atomic_item(scenario: dict, case: str) -> dict:
    """ Builds an item for the given case from the scenario. includes building options"""
    seed = scenario["seed"]

    return {
        "id": f"{scenario['scenario_id']}-{case}",
        "scenario_id": scenario["scenario_id"],
        "case": case,
        "cue": scenario["cue"],
        "context": seed["contexts"][case],
        "utterance": seed["utterance"],
        "options": build_options(seed, case),
        "right_answer": right_answer_for_case(case),
    }


# add labels A,B,C,D
def add_option_labels(item: dict) -> dict:
    option_order = ["object_last", "subject_last", "verb_last", "distractor"]
    labels = ["A", "B", "C", "D"]

    for key, label in zip(option_order, labels):
        item["options"][key]["label"] = label

    return item


def expand_scenario_to_atomic_items(scenario: dict) -> list[dict]:
    cases = ["focus_object", "focus_subject", "focus_verb"]
    return [build_atomic_item(scenario, case) for case in cases]


def build_items_from_scenarios(scenarios: list[dict]) -> list[dict]:
    items = []

    # explode scenarios
    for scenario in scenarios:
        atomic_items = expand_scenario_to_atomic_items(scenario)
        items.extend(atomic_items)

    # add labels
    for item in items:
        item = add_option_labels(item)

    return items


In [14]:
print(f"number of scenarios: {len(all_data)}")

all_items = build_items_from_scenarios(all_data)
print(f"number of items generated from scenarios: {len(all_items)}")

number of scenarios: 240
number of items generated from scenarios: 720


In [15]:
print("example: ")
all_items[0]

example: 


{'id': '17-focus_object',
 'scenario_id': 17,
 'case': 'focus_object',
 'cue': 'baseline',
 'context': 'Z kuchyně bylo cítit, že kuchař sice něco připálil, ale nebylo jasné, o jaký pokrm vlastně šlo.',
 'utterance': 'Marek na to najednou prohlásil:',
 'options': {'object_last': {'structure': 'SVO',
   'sentence': 'Kuchař připálil omáčku.',
   'label': 'A'},
  'subject_last': {'structure': 'OVS',
   'sentence': 'Omáčku připálil kuchař.',
   'label': 'B'},
  'verb_last': {'structure': 'SOV',
   'sentence': 'Kuchař omáčku připálil.',
   'label': 'C'},
  'distractor': {'sentence': 'Kuchař otevřel troubu.', 'label': 'D'}},
 'right_answer': 'object_last'}

In [16]:
path = r"data\information-structure\master\information-structure-all-items-720.json"
with open(path, "w", encoding="utf-8") as f:
        json.dump(all_items, f, ensure_ascii=False, indent=2)

## observe an example item

In [17]:
def render_item_preview(item: dict) -> str:
    lines = []

    lines.append(f"context: {item['context']}")
    lines.append(f"utterance: {item['utterance']}")
    lines.append("options:")

    option_order = ["object_last", "subject_last", "verb_last", "distractor"]

    for key in option_order:
        opt = item["options"][key]

        lines.append(f"- [{opt['label']}]: {opt['sentence']}")

    lines.append(f"\ncorrect answer: {item["options"][item["right_answer"]]["label"]}")

    return "\n".join(lines) 

In [19]:
print(render_item_preview(all_items[130]))

context: Bylo jasné, že ten časopis někdo vyhodil, ale nikdo nevěděl, kdo to ve skutečnosti udělal.
utterance: Ondřej na to najednou prohlásil:
options:
- [A]: Vyhodil uklízeč časopis.
- [B]: Časopis vyhodil uklízeč.
- [C]: Uklízeč časopis vyhodil.
- [D]: Časopis četla recepční.

correct answer: B


## transform to eval format

In [20]:
from __future__ import annotations

from typing import Any


OPTION_TYPES = ["object_last", "subject_last", "verb_last", "distractor"]


def build_information_structure_id(item: dict[str, Any]) -> str:
    """
    build id in the format:
    information-structure-{scenario_id}-{case}

    example:
    information-structure-17-focus_object
    """
    return f"information-structure-{item['scenario_id']}-{item['case']}"


def convert_information_structure_item(item: dict[str, Any]) -> dict[str, Any]:
    """
    convert one information-structure item from the scenario schema
    into the eval schema.
    """
    options_out = []
    label_by_type: dict[str, str] = {}

    for opt_type in OPTION_TYPES:
        if opt_type not in item["options"]:
            raise ValueError(
                f"missing option type {opt_type!r} in item {item.get('id', '<no-id>')!r}"
            )

        opt = item["options"][opt_type]

        if "label" not in opt:
            raise ValueError(
                f"missing label for option type {opt_type!r} in item {item.get('id', '<no-id>')!r}"
            )

        if "sentence" not in opt:
            raise ValueError(
                f"missing sentence for option type {opt_type!r} in item {item.get('id', '<no-id>')!r}"
            )

        label = str(opt["label"]).strip()
        text = opt["sentence"]

        options_out.append(
            {
                "label": label,
                "type": opt_type,
                "text": text,
            }
        )
        label_by_type[opt_type] = label

    right_answer_type = item["right_answer"]
    if right_answer_type not in label_by_type:
        raise ValueError(
            f"right_answer={right_answer_type!r} is not among option types "
            f"in item {item.get('id', '<no-id>')!r}"
        )

    return {
        "id": build_information_structure_id(item),
        "creation_method": "generation",
        "phenomenon": "information_structure",
        "category": item['cue'],
        "context": item["context"],
        "utterance": item["utterance"],
        "options": options_out,
        "gold_label": label_by_type[right_answer_type],
        "metadata": {
            "cue": item["cue"],
            "case": item["case"],
            "scenario_id": item["scenario_id"],
        },
    }

def convert_information_structure_dataset(dataset: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """
    Convert a whole list of information-structure items
    into the eval schema.
    """
    return [convert_information_structure_item(item) for item in dataset]

In [21]:
# test single item
test_item = all_items[0]
convert_information_structure_item(test_item)

{'id': 'information-structure-17-focus_object',
 'creation_method': 'generation',
 'phenomenon': 'information_structure',
 'category': 'baseline',
 'context': 'Z kuchyně bylo cítit, že kuchař sice něco připálil, ale nebylo jasné, o jaký pokrm vlastně šlo.',
 'utterance': 'Marek na to najednou prohlásil:',
 'options': [{'label': 'A',
   'type': 'object_last',
   'text': 'Kuchař připálil omáčku.'},
  {'label': 'B', 'type': 'subject_last', 'text': 'Omáčku připálil kuchař.'},
  {'label': 'C', 'type': 'verb_last', 'text': 'Kuchař omáčku připálil.'},
  {'label': 'D', 'type': 'distractor', 'text': 'Kuchař otevřel troubu.'}],
 'gold_label': 'A',
 'metadata': {'cue': 'baseline', 'case': 'focus_object', 'scenario_id': 17}}

In [22]:
# converting full items dataset

eval_items = convert_information_structure_dataset(all_items)
print(f"length: {len(eval_items)}")
print("example:")
eval_items[666]

length: 720
example:


{'id': 'information-structure-151-focus_object',
 'creation_method': 'generation',
 'phenomenon': 'information_structure',
 'category': 'correction',
 'context': 'Z výpisu z účtu vyšlo najevo, že Michal sice něco zaplatil, ale nikdo nevěděl, o co se jednalo. Martin podotkl, že Michal zaplatil účet.',
 'utterance': 'Eva ho opravila:',
 'options': [{'label': 'A',
   'type': 'object_last',
   'text': 'Michal zaplatil pokutu.'},
  {'label': 'B', 'type': 'subject_last', 'text': 'Pokutu zaplatil Michal.'},
  {'label': 'C', 'type': 'verb_last', 'text': 'Michal pokutu zaplatil.'},
  {'label': 'D', 'type': 'distractor', 'text': 'Michal otevřel aplikaci.'}],
 'gold_label': 'A',
 'metadata': {'cue': 'correction', 'case': 'focus_object', 'scenario_id': 151}}

## save to json

In [24]:
import json

with open("data/information-structure/eval/inf_structure_eval_720.json", "w", encoding="utf-8") as f:
    json.dump(
        eval_items,
        f,
        ensure_ascii=False,
        indent=2
    )

## distractors analysis

In [25]:
# data and file paths
DATA_PATH = Path.cwd() / "data/information-structure/generated"

FILES = [
    "inf-structure-baseline-80-raw.json",
    "inf-structure-explicit-question-80-raw.json",
    "inf-structure-correction-80-raw.json",
]  


In [ ]:
data_baseline = load_dataset(DATA_PATH / "inf-structure-baseline-80-raw.json")
data_exp = load_dataset(DATA_PATH / "inf-structure-explicit-question-80-raw.json")
data_cor = load_dataset(DATA_PATH / "inf-structure-correction-80-raw.json")

In [ ]:
data_baseline[0]

In [ ]:
def print_scenario_contexts(item: dict) -> None:
    """ for a scenario prints context + distractor for all 3 cases"""
    s = ""

    for foc, sen in item['seed']['contexts'].items():
        # printing context
        s += f"\n{sen}\n"

        # utterance is fixed
        s += (item['seed']['utterance'] + "\n")

        # adding corresponding distractor
        s += f"\n{item['seed']['distractor_bank'][foc]}"

        s += "\n\n\n"

    print(s)

print_scenario_contexts(data_baseline[0])